# Chapter 7 — Ordering: learned rankers and list reranking

One unified notebook for the chapter. **All logic lives in `recsys.fourstage_recsys.ordering`** — this notebook only loads, calls, and displays.

Flow: candidates + upstream scores → features → LambdaMART (with the cross-feature ablation) → DCN-v2 → everything evaluated against the **scored-order null hypothesis** → SHAP → the 7.7 ordering pass (MMR + genre cap) with the before/after and ILD.

Set `USE_SYNTHETIC = False` and point `ML25M_DIR` at MovieLens 25M for the real runs that fill the chapter's `[fill]` tables.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import torch

from recsys.fourstage_recsys.ordering import (
    FEATURE_COLS, FEATURE_COLS_NO_CROSS, DCN_DENSE_COLS,
    build_genre_matrix, build_feature_frame, attach_labels, attach_upstream_scores,
    ScoredOrderBaseline, evaluate_ranker, list_divergence, ild_of_list,
    train_lambdamart, explain_ranker, feature_importance,
    make_movie_index, add_movie_index, fit_scaler, make_loaders,
    DCNv2, train_dcn, DCNRanker,
    mmr_rerank, order_stage, candidate_similarity, primary_genre_map,
    make_synthetic_dataset,
)

USE_SYNTHETIC = False          # False -> real MovieLens 25M + chapter-5 outputs
ML25M_DIR = "../data/ml-25m"  # ratings.csv, movies.csv
SEED = 7
np.random.seed(SEED); torch.manual_seed(SEED)

## Data

**Synthetic mode** stands in until the chapter-5 wiring lands. **Real mode** needs two things from the chapter-5 pipeline:

1. `candidates`: one row per (userId, movieId) the retrieval stage returned per user;
2. `scores`: the cross-encoder score for every candidate (`userId, movieId, cross_encoder_score`).

The scored order is the null hypothesis, so the scores are not optional — `attach_upstream_scores` raises if any candidate is missing one.

In [ ]:
if USE_SYNTHETIC:
    train, heldout, movies, candidates = make_synthetic_dataset(seed=SEED)
else:
    ratings = pd.read_csv(f"{ML25M_DIR}/ratings.csv")
    movies = pd.read_csv(f"{ML25M_DIR}/movies.csv")
    cutoff = ratings["timestamp"].quantile(0.8)   # temporal split (ch. 4)
    train = ratings[ratings["timestamp"] <= cutoff]
    heldout = ratings[ratings["timestamp"] > cutoff]

    # --- chapter-5 wiring hook (open item) -------------------------------
    # candidates = pd.read_parquet("../artifacts/ch5_candidates.parquet")
    # scores = pd.read_parquet("../artifacts/ch5_cross_encoder_scores.parquet")
    # candidates = attach_labels(candidates, heldout)
    # candidates = attach_upstream_scores(candidates, scores)
    raise NotImplementedError("wire in chapter-5 candidates + cross-encoder scores")

genre_matrix = build_genre_matrix(movies)
frame = build_feature_frame(candidates, train, genre_matrix)
frame[FEATURE_COLS].describe().T.head(10)

In [ ]:
def split_by_user(frame, test_frac=0.3, seed=SEED):
    users = np.array(sorted(frame["userId"].unique()))
    rng = np.random.default_rng(seed); rng.shuffle(users)
    test_users = set(users[: max(int(len(users) * test_frac), 1)].tolist())
    m = frame["userId"].isin(test_users)
    return frame[~m].reset_index(drop=True), frame[m].reset_index(drop=True)

fit_frame, test_frame = split_by_user(frame)
len(fit_frame), len(test_frame)

## The null hypothesis row

Every result table starts with the scored order — the candidates sorted by the chapter-5 cross-encoder score. `ScoredOrderBaseline` is that order dressed up as a model.

In [ ]:
try:
    import mlflow
    mlflow.set_experiment("ch07-ordering")
    MLFLOW = True
except Exception:
    MLFLOW = False

results = {}
results["Scored order (cross-encoder, ch5)"] = evaluate_ranker(
    test_frame, FEATURE_COLS, ScoredOrderBaseline())
pd.DataFrame(results).T.round(4)

## LambdaMART (7.4) — with the cross-feature ablation (7.6.1)

Two runs: full `FEATURE_COLS`, and `FEATURE_COLS_NO_CROSS` without the hand-engineered `x_*` products. The gap between the two rows is what the feature engineering buys — the thing DCN-v2 will try to learn on its own.

In [ ]:
for name, cols in [("LambdaMART, with cross features", FEATURE_COLS),
                   ("LambdaMART, no cross features", FEATURE_COLS_NO_CROSS)]:
    model = train_lambdamart(fit_frame, cols, n_estimators=600)
    results[name] = evaluate_ranker(test_frame, cols, model)
    if name.endswith("with cross features"):
        lm = model
    if MLFLOW:
        with mlflow.start_run(run_name=name):
            mlflow.log_params({"model": "lambdamart", "n_features": len(cols)})
            mlflow.log_metrics({k.replace("@", "_at_"): v
                                for k, v in results[name].items()})
pd.DataFrame(results).T.round(4)  # -> Table 7.2 (+ ablation rows of Table 7.3)

## SHAP (7.4.3) — Figures 7.3 and 7.4

Held-out sample, not training rows. The beeswarm is Figure 7.3; the dependence plot of `x_genre_affinity` colored by `item_log_pop` is Figure 7.4 (the cross-of-a-cross).

In [ ]:
import shap, matplotlib.pyplot as plt

sample = test_frame.sample(min(2000, len(test_frame)), random_state=SEED)
shap_values = explain_ranker(lm, sample, FEATURE_COLS)          # Figure 7.3
shap.dependence_plot("x_genre_affinity", shap_values,
                     sample[FEATURE_COLS],
                     interaction_index="item_log_pop")           # Figure 7.4
feature_importance(lm, FEATURE_COLS).head(10)

## DCN-v2 (7.5)

Same candidates, same labels, same null — only the model changes. Note `DCN_DENSE_COLS` excludes the hand-built crosses on purpose: the cross network is supposed to find them itself. Evaluation goes through the same `evaluate_ranker`, with `movie_idx` riding along in the feature columns for the embedding lookup.

In [ ]:
movie_index = make_movie_index(frame["movieId"])
fit_idx, test_idx = add_movie_index(fit_frame, movie_index), add_movie_index(test_frame, movie_index)
dcn_fit, dcn_val = split_by_user(fit_idx, test_frac=0.15, seed=11)
scaler = fit_scaler(dcn_fit, DCN_DENSE_COLS)   # training window only
loaders = make_loaders(dcn_fit, dcn_val, DCN_DENSE_COLS, scaler)

dcn_model = DCNv2(n_movies=len(movie_index), dense_dim=len(DCN_DENSE_COLS),
                  emb_dim=32, n_cross=3, mlp_dims=(128, 64))
dcn_model = train_dcn(dcn_model, *loaders, epochs=10, lr=1e-3)
dcn = DCNRanker(dcn_model, scaler, DCN_DENSE_COLS)

results["DCN-v2"] = evaluate_ranker(test_idx, DCN_DENSE_COLS + ["movie_idx"], dcn)
if MLFLOW:
    with mlflow.start_run(run_name="DCN-v2"):
        mlflow.log_params({"model": "dcn_v2", "emb_dim": 32, "n_cross": 3})
        mlflow.log_metrics({k.replace("@", "_at_"): v
                            for k, v in results["DCN-v2"].items()})
pd.DataFrame(results).T.round(4)  # -> Table 7.3

## The ordering pass (7.7) — MMR + genre cap, before and after

The axis changes here: the reranked list should **not** win on NDCG. We report what this stage can honestly self-measure — ILD, composition, how much the list changed — and nothing counterfactual (that's chapter 12's job).

In [ ]:
user_id = test_frame["userId"].iloc[0]
one = test_frame[test_frame["userId"] == user_id]
ranked = one.assign(_s=lm.predict(one[FEATURE_COLS])).sort_values("_s", ascending=False)
cand_ids = ranked["movieId"].tolist()
relevance = ranked["_s"].to_numpy()
sim = candidate_similarity(cand_ids, genre_matrix)
genres = primary_genre_map(genre_matrix)
titles = movies.set_index("movieId")["title"]

before = cand_ids[:10]
after = order_stage(cand_ids, relevance, sim, genres, k=10, lam=0.7, cap=3)

rows = []
for name, lst in [("Ranked order (LambdaMART)", before),
                  ("+ MMR + genre cap", after)]:
    rows.append({
        "list": name,
        "mean relevance (ranker score)": float(np.mean(
            [relevance[cand_ids.index(m)] for m in lst])),
        "ILD@10": ild_of_list(lst, cand_ids, sim),
        "max items per genre": max(pd.Series([genres[m] for m in lst]).value_counts()),
        **{k: v for k, v in list_divergence(
            np.array(cand_ids), np.array(lst), k=10).items()},
    })
display(pd.DataFrame(rows).round(3))  # -> Table 7.4

pd.DataFrame({  # -> Figure 7.6 source: the visible transformation
    "before": [f"{titles.get(m, m)}  [{genres[m]}]" for m in before],
    "after":  [f"{titles.get(m, m)}  [{genres[m]}]" for m in after],
})

### The λ trade-off (Further Reading: the full Pareto sweep)

One picture of the knob: as λ falls, ILD rises and the ranker's mean relevance falls. There is no right setting — it's a product decision.

In [ ]:
lams = np.linspace(0.2, 1.0, 9)
sweep = []
for lam in lams:
    lst = mmr_rerank(cand_ids, relevance, sim, k=10, lam=lam)
    sweep.append({"lam": lam,
                  "ILD@10": ild_of_list(lst, cand_ids, sim),
                  "mean relevance": float(np.mean(
                      [relevance[cand_ids.index(m)] for m in lst]))})
sweep = pd.DataFrame(sweep)
ax = sweep.plot(x="lam", y="ILD@10", marker="o")
sweep.plot(x="lam", y="mean relevance", marker="s", secondary_y=True, ax=ax);

## Filling the chapter

- `results` table → Table 7.2 (baseline + LambdaMART) and Table 7.3 (all rows, ablation included). Record training times alongside.
- SHAP cell → Figures 7.3 / 7.4 (export as PNG).
- Ordering cell → Table 7.4 and Figure 7.6.
- Never report an offline NDCG lift for the 7.7 transforms — by design they trade relevance for diversity.